In [1]:
%pip -q install rdkit pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 43.4 MB/s eta 0:00:00


In [3]:
import io, json, zipfile, hashlib, shutil, sys
from pathlib import Path
from datetime import datetime, timezone
from itertools import product

import pandas as pd
from rdkit import Chem, rdBase
from rdkit.Chem import Draw, rdMolDescriptors
from google.colab import files
from IPython.display import display

print("Upload: pilot ZIP, Gao reference CSV, Yang validation CSV")
uploads = files.upload()

gao_name = "Gao2021_reference_library_A1_A12.csv"
yang_name = "Yang2019_reference_validation.csv"
assert gao_name in uploads and yang_name in uploads

gao = pd.read_csv(io.BytesIO(uploads[gao_name]))
yang = pd.read_csv(io.BytesIO(uploads[yang_name]))
assert len(gao) == len(yang) == 12
assert yang["Validation_status"].eq("PASS").all()

pilot_zips = [
    name for name in uploads
    if name.startswith("tryptanthrin_library_") and name.endswith(".zip")
]
assert len(pilot_zips) == 1, "Upload your original 30-structure pilot ZIP."

with zipfile.ZipFile(io.BytesIO(uploads[pilot_zips[0]])) as z:
    pilot = pd.read_csv(io.BytesIO(z.read("analog_id_registry.csv")))

PARENT = "O=C1c2ccccc2-n2c1nc1ccccc1c2=O"
parent = Chem.MolFromSmiles(PARENT)

# Applies only to the exact parent SMILES above.
sites = {
    "C-1": 15, "C-2": 14, "C-3": 13, "C-4": 12,
    "C-7": 3, "C-8": 4, "C-9": 5, "C-10": 6
}
fragments = {
    "H": None, "F": "F", "Cl": "Cl", "Br": "Br", "I": "I",
    "CH3": "C", "OCH3": "OC", "NO2": "[N+](=O)[O-]",
    "NH2": "N", "CF3": "C(F)(F)F", "N(CH3)2": "N(C)C"
}

def canon(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES")
    Chem.SanitizeMol(mol)
    return Chem.MolToSmiles(mol, canonical=True)

def build(pattern):
    mol = Chem.Mol(parent)
    for site, group in pattern.items():
        if group == "H":
            continue
        index = sites[site]
        assert parent.GetAtomWithIdx(index).GetTotalNumHs() == 1
        fragment = Chem.MolFromSmiles(fragments[group])
        offset = mol.GetNumAtoms()
        edit = Chem.RWMol(Chem.CombineMols(mol, fragment))
        edit.AddBond(index, offset, Chem.BondType.SINGLE)
        mol = edit.GetMol()
    Chem.SanitizeMol(mol)
    return mol

# Multiple source records can point to the same structure.
evidence = []

def add_evidence(smiles, ref_id, doi, location):
    evidence.append({
        "SMILES": canon(smiles), "Reference_ID": ref_id,
        "Source_DOI": doi, "Source_location": location
    })

for row in gao.to_dict("records"):
    add_evidence(row["SMILES"], row["Reference_ID"],
                 row["Source_DOI"], row["Source_location"])

for row in yang.to_dict("records"):
    add_evidence(row["SMILES"], row["Reference_ID"],
                 row["Source_DOI"], "Author-supplied supplementary CSV")

yang13 = {
    "5a": {}, "5b": {"C-8": "CH3"}, "5c": {"C-8": "F"},
    "5d": {"C-8": "Br"}, "5e": {"C-2": "F"},
    "5f": {"C-2": "F", "C-8": "F"},
    "5g": {"C-7": "Cl"}, "5h": {"C-9": "Cl"},
    "5i": {"C-8": "NO2"}
}
zou23 = {
    "A1": {"C-1": "F"}, "A2": {"C-2": "F"},
    "A3": {"C-2": "Cl"}, "A4": {"C-2": "Br"},
    "A5": {"C-2": "CH3"}, "A6": {"C-2": "OCH3"},
    "A7": {"C-2": "NO2"}, "A8": {"C-1": "Cl"},
    "A9": {"C-2": "F", "C-3": "F"},
    "B1": {"C-10": "F"}, "B2": {"C-8": "F"},
    "B3": {"C-8": "Cl"}, "B4": {"C-8": "Br"},
    "B5": {"C-8": "CH3"}, "B6": {"C-8": "OCH3"},
    "B7": {"C-8": "NO2"}, "B8": {"C-7": "Cl"},
    "B9": {"C-8": "F", "C-9": "F"}
}

for label, pattern in yang13.items():
    add_evidence(Chem.MolToSmiles(build(pattern)),
                 f"Yang2013_{label}", "10.1021/jm401195n",
                 "Supplement Scheme 1, page 3")

for label, pattern in zou23.items():
    add_evidence(Chem.MolToSmiles(build(pattern)),
                 f"Zou2023_{label}", "10.3390/ijms24021450",
                 "Figure 3; supplement compound names, pages 4-9")

sources = pd.DataFrame(evidence).drop_duplicates()
source_lookup = sources.groupby("SMILES")["Reference_ID"].agg(list).to_dict()

# Design: 77 combinations + seven positional references + 12 supplied structures.
c2 = ["H", "F", "Cl", "Br", "CH3", "OCH3", "NO2"]
c8 = ["H", "F", "Cl", "Br", "I", "CH3", "OCH3",
      "NO2", "NH2", "CF3", "N(CH3)2"]

jobs = [
    ("C2_C8_grid", {"C-2": a, "C-8": b}, None)
    for a, b in product(c2, c8)
]
extras = [
    {"C-1": "F"}, {"C-1": "Cl"}, {"C-7": "Cl"},
    {"C-9": "Cl"}, {"C-10": "F"},
    {"C-2": "F", "C-3": "F"},
    {"C-8": "F", "C-9": "F"}
]
jobs += [("Positional_reference", p, None) for p in extras]
jobs += [
    (r["Reference_ID"], None, r["SMILES"])
    for r in yang.to_dict("records")
]
assert len(jobs) == 96

old_ids = dict(zip(pilot["SMILES"].map(canon), pilot["Analog_ID"]))
assert len(old_ids) == len(pilot)

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
out = Path("expanded_library_runs") / run_id
out.mkdir(parents=True, exist_ok=False)

raw_rows, seen, mols = [], {}, {}
for number, (origin, pattern, supplied) in enumerate(jobs, 1):
    record = {
        "Attempt_ID": f"ATT_{number:04d}", "Generation_origin": origin,
        "Generation_pattern": json.dumps(pattern or {}, sort_keys=True),
        "Analog_ID": "", "SMILES": "", "InChIKey": "",
        "Modified_site": "", "Substituent": "",
        "Modification_type": "Ring substitution or supplied reference",
        "Molecular_formula": "", "Category": "",
        "Reference_IDs": "", "Validation_status": "FAILED",
        "Reason": ""
    }
    try:
        mol = Chem.MolFromSmiles(supplied) if supplied else build(pattern)
        if mol is None:
            raise ValueError("Parsing failed")
        Chem.SanitizeMol(mol)
        smiles = canon(Chem.MolToSmiles(mol))

        if len(Chem.GetMolFrags(mol)) != 1:
            raise ValueError("Disconnected molecule")
        if Chem.GetFormalCharge(mol) != 0:
            raise ValueError("Unexpected net charge")
        match = mol.GetSubstructMatch(parent)
        if not match:
            raise ValueError("Parent scaffold missing")

        # Record complete substituent fragments outside the matched core.
        core = set(match)
        modified, groups = [], []
        for site, index in sites.items():
            center = mol.GetAtomWithIdx(match[index])
            for neighbor in center.GetNeighbors():
                if neighbor.GetIdx() in core:
                    continue
                found, queue = set(), [neighbor.GetIdx()]
                while queue:
                    atom_index = queue.pop()
                    if atom_index in core or atom_index in found:
                        continue
                    found.add(atom_index)
                    queue.extend(
                        n.GetIdx()
                        for n in mol.GetAtomWithIdx(atom_index).GetNeighbors()
                    )
                fragment_text = Chem.MolFragmentToSmiles(
                    mol, atomsToUse=sorted(found), canonical=True
                )
                modified.append(site)
                groups.append(f"{site}:{fragment_text}")

        ref_ids = source_lookup.get(smiles, [])
        analog_id = old_ids.get(smiles)
        if analog_id is None:
            analog_id = "TRY_" + hashlib.sha256(
                smiles.encode()
            ).hexdigest()[:16]

        category = (
            "Parent" if smiles == canon(PARENT)
            else "Published_reference" if ref_ids
            else "Exploratory_combination"
        )
        record.update({
            "Analog_ID": analog_id, "SMILES": smiles,
            "InChIKey": Chem.MolToInchiKey(mol),
            "Modified_site": "; ".join(modified) or "None",
            "Substituent": "; ".join(groups) or "None",
            "Molecular_formula": rdMolDescriptors.CalcMolFormula(mol),
            "Category": category, "Reference_IDs": "; ".join(ref_ids)
        })
        if smiles in seen:
            record["Validation_status"] = "DUPLICATE"
            record["Reason"] = f"Same structure as {seen[smiles]}"
        else:
            seen[smiles] = record["Attempt_ID"]
            record["Validation_status"] = "PASS"
            mols[analog_id] = mol
    except Exception as error:
        record["Reason"] = str(error)
    raw_rows.append(record)

raw = pd.DataFrame(raw_rows)
clean = raw.loc[raw["Validation_status"] == "PASS"].copy()
assert clean["Analog_ID"].is_unique
assert clean["SMILES"].is_unique
assert clean["InChIKey"].is_unique

# Every old pilot structure must survive with its original identifier.
new_ids = clean.set_index("SMILES")["Analog_ID"].to_dict()
assert all(new_ids.get(s) == a for s, a in old_ids.items()), \
    "Pilot structure missing or ID changed"

missing_refs = sorted(set(sources["SMILES"]) - set(clean["SMILES"]))
summary = {
    "Attempted": len(raw),
    "Failed": int(raw["Validation_status"].eq("FAILED").sum()),
    "Duplicates": int(raw["Validation_status"].eq("DUPLICATE").sum()),
    "Retained": len(clean),
    **clean["Category"].value_counts().to_dict(),
    "Missing_reference_structures": len(missing_refs)
}
assert summary["Attempted"] == (
    summary["Failed"] + summary["Duplicates"] + summary["Retained"]
)

raw.to_csv(out / "tryptanthrin_analogs_raw.csv", index=False)
clean.to_csv(out / "tryptanthrin_analogs_clean.csv", index=False)
raw[["Attempt_ID", "Analog_ID", "Validation_status", "Reason"]].to_csv(
    out / "validation_audit.csv", index=False
)
clean[["Analog_ID", "SMILES", "InChIKey"]].to_csv(
    out / "analog_id_registry.csv", index=False
)
sources.to_csv(out / "reference_evidence.csv", index=False)
pd.DataFrame([summary]).to_csv(out / "library_statistics.csv", index=False)

# Save the full attempted design as an editable rule manifest.
raw[["Attempt_ID", "Generation_origin", "Generation_pattern"]].to_csv(
    out / "generation_manifest.csv", index=False
)

with Chem.SDWriter(str(out / "expanded_library.sdf")) as writer:
    for row in clean.to_dict("records"):
        mol = Chem.Mol(mols[row["Analog_ID"]])
        mol.SetProp("_Name", row["Analog_ID"])
        mol.SetProp("Category", row["Category"])
        writer.write(mol)

for start in range(0, len(clean), 12):
    batch = clean.iloc[start:start + 12]
    grid = Draw.MolsToGridImage(
        [mols[a] for a in batch["Analog_ID"]],
        legends=[
            f'{r.Attempt_ID}: {r.Category}'
            for r in batch.itertuples()
        ],
        molsPerRow=3, subImgSize=(400, 300),
        useSVG=False, returnPNG=False
    )
    grid.save(out / f"structures_{start + 1:03d}.png")

# Preserve exact uploaded inputs.
inputs = out / "inputs"
inputs.mkdir()
for name, content in uploads.items():
    (inputs / Path(name).name).write_bytes(content)

metadata = {
    "Run_UTC": run_id,
    "RDKit_version": rdBase.rdkitVersion,
    "pandas_version": pd.__version__,
    "Python": sys.version,
    "Parent_SMILES": PARENT,
    "Site_indices": sites,
    "Manual_structure_review": "PENDING",
    "Design": "77 C2/C8 combinations + 7 positional references + 12 Yang2019 structures",
    "Limitations": [
        "Cross-combinations are exploratory hypotheses.",
        "Published classification refers only to reviewed source matches.",
        "Novelty, activity and synthetic feasibility are not established.",
        "Zou2023 A5 analytical discrepancy requires a source-quality flag."
    ]
}
(out / "run_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)

display(pd.DataFrame([summary]))
print("Pilot IDs preserved:", len(old_ids))
print("Missing reference structures:", len(missing_refs))

archive = shutil.make_archive(
    f"/content/expanded_tryptanthrin_library_{run_id}",
    "zip", root_dir=str(out)
)
files.download(archive)

Upload: pilot ZIP, Gao reference CSV, Yang validation CSV


Saving Yang2019_reference_validation.csv to Yang2019_reference_validation.csv
Saving tryptanthrin_library_20261020T042006129420Z.zip to tryptanthrin_library_20261020T042006129420Z (1).zip
Saving Gao2021_reference_library_A1_A12.csv to Gao2021_reference_library_A1_A12.csv


,Attempted,Failed,Duplicates,Retained,Exploratory_combination,Published_reference,Parent,Missing_reference_structures
0,96,0,0,96,58,37,1,0


Pilot IDs preserved: 30
Missing reference structures: 0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>